# Notebook 01: UGRansome2024 Data Cleaning

**Purpose:** Load the raw UGRansome2024 CSV, remove irrelevant columns, eliminate duplicates and missing values, and encode binary labels.

**Output:** `data/processed/ugr_clean.csv`

In [1]:
import os
os.chdir('/home/elious/research_projects/mdpi_sensors_2026')

import pandas as pd
import numpy as np

RAW_PATH = 'data/raw/UGRansome_Dataset_2024.csv'
OUT_PATH  = 'data/processed/ugr_clean.csv'

df = pd.read_csv(RAW_PATH)
print(f'Raw shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'\nLabel counts (raw):')
print(df['Prediction'].value_counts())

Raw shape: (111904, 14)
Columns: ['Time', 'Protocol', 'Flag', 'Family', 'Clusters', 'SeedAddress', 'ExpAddress', 'BTC', 'USD', 'Netflow_Bytes', 'IPaddress', 'Threats', 'Port', 'Prediction']

Label counts (raw):
Prediction
S     51738
SS    34158
A     26008
Name: count, dtype: int64


In [2]:
# Drop columns not used in the study
COLS_DROP = ['BTC', 'SeedAddress', 'ExpAddress']
df = df.drop(columns=COLS_DROP, errors='ignore')
print(f'After dropping {COLS_DROP}: {df.shape}')

After dropping ['BTC', 'SeedAddress', 'ExpAddress']: (111904, 11)


In [3]:
# Remove duplicate rows
before = len(df)
df = df.drop_duplicates()
print(f'Duplicate rows removed: {before - len(df)}')
print(f'Shape after dedup: {df.shape}')

Duplicate rows removed: 22045
Shape after dedup: (89859, 11)


In [4]:
# Drop rows with missing values
before = len(df)
missing_total = df.isna().sum().sum()
print(f'Total missing cells: {missing_total}')
df = df.dropna()
print(f'Rows dropped for NaN: {before - len(df)}')
print(f'Shape after NaN drop: {df.shape}')

Total missing cells: 0
Rows dropped for NaN: 0
Shape after NaN drop: (89859, 11)


In [5]:
# Encode labels: S and SS -> 0 (benign), A -> 1 (attack)
LABEL_MAP = {'S': 0, 'SS': 0, 'A': 1}
df['Prediction'] = df['Prediction'].map(LABEL_MAP)

assert df['Prediction'].isna().sum() == 0, 'Unmapped labels found'

print(f'Label distribution after encoding:')
print(df['Prediction'].value_counts())
print(f'\nAttack rate: {df["Prediction"].mean():.4f}')

Label distribution after encoding:
Prediction
0    69376
1    20483
Name: count, dtype: int64

Attack rate: 0.2279


In [6]:
# Save cleaned data
df.to_csv(OUT_PATH, index=False)
print(f'Saved: {OUT_PATH}')
print(f'Final shape: {df.shape}')

Saved: data/processed/ugr_clean.csv
Final shape: (89859, 11)


In [7]:
# Append cleaning results to dataset audit
benign_n = int((df['Prediction'] == 0).sum())
attack_n = int((df['Prediction'] == 1).sum())
total_n  = len(df)
audit_entry = (
    f'\n## UGRansome2024 Cleaning (Notebook 01)\n'
    f'- Rows after cleaning: {total_n}\n'
    f'- Benign (0): {benign_n}\n'
    f'- Attack (1): {attack_n}\n'
)
with open('audit/dataset_audit.md', 'a') as f:
    f.write(audit_entry)
print(audit_entry)


## UGRansome2024 Cleaning (Notebook 01)
- Rows after cleaning: 89859
- Benign (0): 69376
- Attack (1): 20483



## Summary

**Purpose:** Clean the raw UGRansome2024 ransomware traffic dataset.

**Method:** Dropped three non-predictive identifier columns (BTC address, seed address, exploit address). Removed duplicate rows. Dropped rows with missing values. Encoded the three-class Prediction column to binary: S and SS map to 0 (benign), A maps to 1 (attack).

**Key findings:** See printed output above for final row count and class distribution.